In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir /
    "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Train: 32337
Validation: 10952
Test: 10840


In [6]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [7]:
train_dataset = SalinasPatchDataset(
    scene=scene,
    indices=train_indices,
    patch_size=15,
    normalize=True,
)

val_dataset = SalinasPatchDataset(
    scene=scene,
    indices=val_indices,
    patch_size=15,
    normalize=True,
)

test_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=15,
    normalize=True,
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

32337 10952 10840


In [8]:
from pathlib import Path
import subprocess

# ============================================================
# GAHT REPOSITORY
# ============================================================

gaht_dir = PROJECT_ROOT / "external" / "gaht"
gaht_dir.mkdir(
    parents=True,
    exist_ok=True,
)

repo_dir = gaht_dir / "Group-Aware-Hierarchical-Transformer"

if not repo_dir.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/MeiShaohui/Group-Aware-Hierarchical-Transformer.git",
            str(repo_dir),
        ],
        check=True,
    )

print("GAHT repository:")
print(repo_dir)

GAHT repository:
c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\external\gaht\Group-Aware-Hierarchical-Transformer


In [9]:
import os

for root, dirs, files in os.walk(repo_dir):

    # don't dump huge git directory
    dirs[:] = [
        d for d in dirs
        if d != ".git"
    ]

    level = root.replace(str(repo_dir), "").count(os.sep)

    if level > 2:
        continue

    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

Group-Aware-Hierarchical-Transformer/
    eval.py
    main.py
    README.md
    requirements.txt
    train.py
    models/
        ablstm.py
        cnn3d.py
        dffn.py
        get_model.py
        m3ddcnn.py
        proposed.py
        rssan.py
        speformer.py
        ssftt.py
        __init__.py
    utils/
        dataset.py
        scheduler.py
        tif2mat.py
        utils.py


In [10]:
import sys

if str(repo_dir) not in sys.path:
    sys.path.insert(
        0,
        str(repo_dir),
    )

print("GAHT path added:")
print(repo_dir)

GAHT path added:
c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\external\gaht\Group-Aware-Hierarchical-Transformer


In [11]:
# ============================================================
# INSPECT GAHT MODEL
# ============================================================

from pathlib import Path

gaht_model_file = (
    repo_dir
    / "models"
    / "proposed.py"
)

gaht_get_model_file = (
    repo_dir
    / "models"
    / "get_model.py"
)

print("=" * 70)
print("GAHT proposed.py")
print("=" * 70)

with open(gaht_model_file, "r", encoding="utf-8") as f:
    proposed_code = f.read()

print(proposed_code)

print()
print("=" * 70)
print("GAHT get_model.py")
print("=" * 70)

with open(gaht_get_model_file, "r", encoding="utf-8") as f:
    get_model_code = f.read()

print(get_model_code)

GAHT proposed.py
import math
import torch
import torch.nn as nn


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        """
        input: (B, N, C)
        B = Batch size, N = patch_size * patch_size, C = dimension hidden_features and out_features
        output: (B, N, C)
        """
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    def __init__(self, dim, num_heads=16, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.

In [12]:
# ============================================================
# INSPECT OFFICIAL GAHT TRAINING CONFIG
# ============================================================

gaht_train_file = repo_dir / "train.py"
gaht_main_file = repo_dir / "main.py"

for file_path in [gaht_train_file, gaht_main_file]:

    print()
    print("=" * 70)
    print(file_path.name)
    print("=" * 70)

    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    print(code)


train.py
import os
import numpy as np
from tqdm import tqdm
import torch
from utils.utils import grouper, sliding_window, count_sliding_window


def train(network, optimizer, criterion, train_loader, val_loader, epoch, saving_path, device, scheduler=None):

    best_acc = -0.1
    losses = []

    for e in tqdm(range(1, epoch+1), desc="training the network"):
        network.train()
        for batch_idx, (images, targets) in enumerate(train_loader):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = network(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if e % 10 == 0 or e == 1:
            mean_losses = np.mean(losses)
            train_info = "train at epoch {}/{}, loss={:.6f}"
            train_info = train_info.format(e, epoch,  mean_losses)
            tqdm.write(train_info)
            losses = [

In [13]:
# ============================================================
# GAHT IMPORT
# ============================================================

import sys
import torch
import torch.nn as nn

# Make sure GAHT repo is first in path
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from models.proposed import proposed

print("GAHT proposed model imported successfully.")

GAHT proposed model imported successfully.


In [14]:
# ============================================================
# OFFICIAL GAHT - SALINAS
# ============================================================

gaht_model = proposed(
    dataset="sa",
    patch_size=15,
)

print(gaht_model)

MyTransformer(
  (pad): ReplicationPad3d((0, 0, 0, 0, 0, 4))
  (patch_embed1): GroupedPixelEmbedding(
    (proj): Conv2d(208, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
    (batch_norm): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
  )
  (block1): ModuleList(
    (0-1): 2 x Block(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (q): Linear(in_features=256, out_features=256, bias=True)
        (kv): Linear(in_features=256, out_features=512, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=256, out_features=256, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=256, out_features=256, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_f

In [15]:
def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


gaht_parameters = count_trainable_parameters(
    gaht_model
)

print()
print("=" * 60)
print("GAHT MODEL INFORMATION")
print("=" * 60)
print(
    "Trainable parameters:",
    gaht_parameters
)
print(
    "Parameters (M):",
    gaht_parameters / 1e6
)


GAHT MODEL INFORMATION
Trainable parameters: 972624
Parameters (M): 0.972624


In [16]:
# ============================================================
# GAHT DATASET ADAPTER
# ============================================================

class GAHTDataset(torch.utils.data.Dataset):

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):

        x, y = self.base_dataset[idx]

        # Convert numpy -> tensor if necessary
        if not torch.is_tensor(x):
            x = torch.from_numpy(
                x
            )

        x = x.float()

        # Expected by GAHT:
        # [C,H,W] -> [1,C,H,W]
        if x.ndim == 3:
            x = x.unsqueeze(0)

        elif x.ndim != 4:
            raise ValueError(
                f"Unexpected input shape: {x.shape}"
            )

        y = torch.as_tensor(
            y,
            dtype=torch.long
        )

        return x, y

In [17]:
gaht_train_dataset = GAHTDataset(
    train_dataset
)

gaht_val_dataset = GAHTDataset(
    val_dataset
)

gaht_test_dataset = GAHTDataset(
    test_dataset
)

print(
    len(gaht_train_dataset),
    len(gaht_val_dataset),
    len(gaht_test_dataset)
)

32337 10952 10840


In [18]:
GAHT_BATCH_SIZE = 128

gaht_train_loader = torch.utils.data.DataLoader(
    gaht_train_dataset,
    batch_size=GAHT_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

gaht_val_loader = torch.utils.data.DataLoader(
    gaht_val_dataset,
    batch_size=GAHT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

gaht_test_loader = torch.utils.data.DataLoader(
    gaht_test_dataset,
    batch_size=GAHT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("GAHT loaders created.")

GAHT loaders created.


In [19]:
x, y = next(
    iter(gaht_train_loader)
)

print("=" * 60)
print("GAHT INPUT CHECK")
print("=" * 60)

print("Input :", x.shape)
print("Labels:", y.shape)
print("dtype :", x.dtype)
print("label dtype:", y.dtype)

GAHT INPUT CHECK
Input : torch.Size([128, 1, 204, 15, 15])
Labels: torch.Size([128])
dtype : torch.float32
label dtype: torch.int64


In [20]:
gaht_model = gaht_model.to(device)
gaht_model.eval()

with torch.no_grad():
    x_device = x.to(device)

    logits = gaht_model(
        x_device
    )

print()
print("=" * 60)
print("GAHT FORWARD CHECK")
print("=" * 60)

print("Input :", x_device.shape)
print("Output:", logits.shape)


GAHT FORWARD CHECK
Input : torch.Size([128, 1, 204, 15, 15])
Output: torch.Size([128, 16])


In [22]:
# ============================================================
# GAHT: CHECK GPU
# ============================================================

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        torch.cuda.memory_allocated() / 1024**2,
        "MB"
    )
    print(
        "GPU memory reserved:",
        torch.cuda.memory_reserved() / 1024**2,
        "MB"
    )

Device: cuda
GPU: NVIDIA GeForce RTX 2050
GPU memory allocated: 75.9521484375 MB
GPU memory reserved: 2368.0 MB


In [23]:
# ============================================================
# GAHT AMP SETUP
# ============================================================

import torch

use_amp = torch.cuda.is_available()

if use_amp:
    amp_dtype = torch.float16
    scaler = torch.amp.GradScaler("cuda")
else:
    amp_dtype = torch.float32
    scaler = None

print("AMP enabled:", use_amp)
print("AMP dtype  :", amp_dtype)

AMP enabled: True
AMP dtype  : torch.float16


In [24]:
# ============================================================
# GAHT TRAINING — AMP VERSION
# ============================================================

def train_gaht_one_epoch_amp(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler,
):

    model.train()

    running_loss = 0.0
    total_samples = 0

    batch_bar = tqdm(
        loader,
        desc="  Train",
        leave=False,
        dynamic_ncols=True,
    )

    for x, y in batch_bar:

        x = x.to(
            device,
            non_blocking=True,
        )

        y = y.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # MIXED PRECISION FORWARD
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):

            logits = model(x)

            loss = criterion(
                logits,
                y,
            )

        # ----------------------------------------------------
        # SCALED BACKWARD
        # ----------------------------------------------------

        if scaler is not None:

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

        else:

            loss.backward()

            optimizer.step()

        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        batch_size = y.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

        avg_loss = (
            running_loss /
            total_samples
        )

        batch_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{avg_loss:.4f}",
        )

    return (
        running_loss /
        total_samples
    )

In [25]:
# ============================================================
# GAHT AMP — ONE EPOCH SPEED TEST
# ============================================================

import time

torch.cuda.synchronize()

start_time = time.perf_counter()

test_loss = train_gaht_one_epoch_amp(
    gaht_model,
    gaht_train_loader,
    criterion,
    optimizer,
    device,
    scaler,
)

torch.cuda.synchronize()

elapsed = (
    time.perf_counter() -
    start_time
)

print()
print("=" * 70)
print("GAHT AMP SPEED TEST")
print("=" * 70)

print(
    f"Training loss : {test_loss:.4f}"
)

print(
    f"Epoch time    : {elapsed:.2f} sec"
)

print(
    f"Epoch time    : {elapsed / 60:.2f} min"
)

print(
    f"Estimated 50 epochs : "
    f"{elapsed * 50 / 60:.1f} min"
)

print("=" * 70)


GAHT AMP SPEED TEST
Training loss : 0.0424
Epoch time    : 91.01 sec
Epoch time    : 1.52 min
Estimated 50 epochs : 75.8 min


In [26]:
gaht_model = proposed(
    dataset="sa",
    patch_size=15,
).to(device)

print(
    "GAHT parameters:",
    count_trainable_parameters(gaht_model)
)

GAHT parameters: 972624


In [27]:
import torch
import torch.nn as nn
from copy import deepcopy
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    gaht_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer:", type(optimizer).__name__)
print("Scheduler:", type(scheduler).__name__)
print("AMP: enabled")

Optimizer: AdamW
Scheduler: CosineAnnealingLR
AMP: enabled


In [28]:
def train_gaht_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler,
):
    model.train()

    running_loss = 0.0
    total_samples = 0

    batch_bar = tqdm(
        loader,
        desc="  Train",
        leave=False,
        dynamic_ncols=True,
    )

    for x, y in batch_bar:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = y.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

        avg_loss = running_loss / total_samples

        batch_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{avg_loss:.4f}",
        )

    return running_loss / total_samples

In [30]:
val_acc, val_f1 = evaluate_gaht(
    gaht_model,
    gaht_val_loader,
    device,
)

print("Validation accuracy :", val_acc)
print("Validation Macro-F1  :", val_f1)

  Val  :   0%|          | 0/86 [00:00<?, ?it/s]

Validation accuracy : 0.015339663988312637
Validation Macro-F1  : 0.01580045798969302


In [31]:
EPOCHS = 50

best_gaht_val_f1 = -1.0
best_gaht_epoch = 0
best_gaht_state = None

gaht_history = []

epoch_bar = tqdm(
    range(1, EPOCHS + 1),
    desc="GAHT Epochs",
    dynamic_ncols=True,
)

for epoch in epoch_bar:

    train_loss = train_gaht_one_epoch(
        gaht_model,
        gaht_train_loader,
        criterion,
        optimizer,
        device,
        scaler,
    )

    val_acc, val_f1 = evaluate_gaht(
        gaht_model,
        gaht_val_loader,
        device,
    )

    scheduler.step()

    is_best = val_f1 > best_gaht_val_f1

    if is_best:
        best_gaht_val_f1 = val_f1
        best_gaht_epoch = epoch

        best_gaht_state = {
            k: v.detach().cpu().clone()
            for k, v in gaht_model.state_dict().items()
        }

    gaht_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    epoch_bar.set_postfix(
        loss=f"{train_loss:.4f}",
        val_acc=f"{val_acc:.4f}",
        val_f1=f"{val_f1:.4f}",
        best=f"{best_gaht_val_f1:.4f}",
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f} | "
        f"Best F1: {best_gaht_val_f1:.4f}"
        + ("  ★ BEST" if is_best else ""),
        flush=True,
    )

print()
print("=" * 70)
print("GAHT TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", best_gaht_epoch)
print(
    "Best validation Macro-F1:",
    best_gaht_val_f1,
)

GAHT Epochs:   0%|          | 0/50 [01:48<?, ?it/s, best=0.8668, loss=0.1913, val_acc=0.8986, val_f1=0.8668]

Epoch 01/50 | Loss: 0.1913 | Val Acc: 0.8986 | Val Macro-F1: 0.8668 | Best F1: 0.8668  ★ BEST


GAHT Epochs:   2%|▏         | 1/50 [04:26<1:28:43, 108.65s/it, best=0.8789, loss=0.0439, val_acc=0.8827, val_f1=0.8789]

Epoch 02/50 | Loss: 0.0439 | Val Acc: 0.8827 | Val Macro-F1: 0.8789 | Best F1: 0.8789  ★ BEST


GAHT Epochs:   4%|▍         | 2/50 [07:37<1:50:05, 137.61s/it, best=0.8789, loss=0.0399, val_acc=0.8482, val_f1=0.8250]

...

Epoch 49/50 | Loss: 0.0001 | Val Acc: 0.8530 | Val Macro-F1: 0.8460 | Best F1: 0.8858


GAHT Epochs:  98%|█████████▊| 49/50 [2:16:41<02:46, 166.95s/it, best=0.8858, loss=0.0001, val_acc=0.8508, val_f1=0.8506]

Epoch 50/50 | Loss: 0.0001 | Val Acc: 0.8508 | Val Macro-F1: 0.8506 | Best F1: 0.8858


GAHT Epochs: 100%|██████████| 50/50 [2:16:41<00:00, 164.04s/it, best=0.8858, loss=0.0001, val_acc=0.8508, val_f1=0.8506]


GAHT TRAINING COMPLETE
Best epoch: 21
Best validation Macro-F1: 0.8857852943726179


In [32]:
# ============================================================
# RESTORE BEST GAHT MODEL
# ============================================================

gaht_model.load_state_dict(best_gaht_state)
gaht_model = gaht_model.to(device)
gaht_model.eval()

print("Best GAHT epoch:", best_gaht_epoch)
print("Best validation Macro-F1:", best_gaht_val_f1)

Best GAHT epoch: 21
Best validation Macro-F1: 0.8857852943726179


In [33]:
# ============================================================
# GAHT SALINAS TEST EVALUATION
# ============================================================

gaht_model.load_state_dict(best_gaht_state)
gaht_model = gaht_model.to(device)
gaht_model.eval()

test_acc, test_f1 = evaluate_gaht(
    gaht_model,
    gaht_test_loader,
    device,
)

print("=" * 60)
print("GAHT SALINAS TEST RESULTS")
print("=" * 60)
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")

GAHT SALINAS TEST RESULTS
Accuracy : 0.9581
Macro-F1 : 0.9424


In [34]:
# ============================================================
# GAHT EFFICIENCY BENCHMARK
# ============================================================

import time
import torch
from thop import profile


# ------------------------------------------------------------
# 1. MODEL SIZE / PARAMETERS
# ------------------------------------------------------------

gaht_model.eval()
gaht_model = gaht_model.to(device)

gaht_parameters = sum(
    p.numel()
    for p in gaht_model.parameters()
    if p.requires_grad
)

gaht_parameters_m = gaht_parameters / 1e6


# ------------------------------------------------------------
# 2. RAW STATE-DICT SIZE
# ------------------------------------------------------------

gaht_state_dict_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in gaht_model.state_dict().values()
)

gaht_model_size_mb = (
    gaht_state_dict_bytes / (1024 ** 2)
)


print("Trainable parameters:", gaht_parameters)
print("Parameters (M):", gaht_parameters_m)
print(
    f"Raw state-dict size: "
    f"{gaht_model_size_mb:.3f} MB"
)


# ------------------------------------------------------------
# 3. FLOPs
# ------------------------------------------------------------

gaht_model.eval()

dummy_input = torch.randn(
    1,
    1,
    204,
    15,
    15,
    device=device,
)

with torch.no_grad():

    gaht_flops, gaht_params_thop = profile(
        gaht_model,
        inputs=(dummy_input,),
        verbose=False,
    )

gaht_gflops = gaht_flops / 1e9

print("FLOPs   :", gaht_flops)
print("GFLOPs  :", gaht_gflops)


# ------------------------------------------------------------
# 4. GPU MEMORY
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():
    _ = gaht_model(dummy_input)

torch.cuda.synchronize()

gaht_peak_gpu_memory_mb = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 2)
)

print(
    "Peak GPU memory:",
    gaht_peak_gpu_memory_mb,
    "MB"
)


# ------------------------------------------------------------
# 5. LATENCY
# ------------------------------------------------------------

BENCH_BATCH_SIZE = 64
WARMUP = 20
ITERATIONS = 100

benchmark_input = torch.randn(
    BENCH_BATCH_SIZE,
    1,
    204,
    15,
    15,
    device=device,
)


# Warm-up
with torch.no_grad():

    for _ in range(WARMUP):
        _ = gaht_model(benchmark_input)

torch.cuda.synchronize()


# Timed inference
times_ms = []

with torch.no_grad():

    for _ in range(ITERATIONS):

        torch.cuda.synchronize()

        start = time.perf_counter()

        _ = gaht_model(
            benchmark_input
        )

        torch.cuda.synchronize()

        end = time.perf_counter()

        times_ms.append(
            (end - start) * 1000
        )


times_ms = torch.tensor(
    times_ms,
    dtype=torch.float64,
)

gaht_latency_batch_ms = times_ms.mean().item()
gaht_latency_batch_median_ms = times_ms.median().item()

gaht_latency_batch_p95_ms = torch.quantile(
    times_ms,
    0.95,
).item()

gaht_latency_batch_std_ms = times_ms.std().item()

gaht_latency_per_sample_ms = (
    gaht_latency_batch_ms /
    BENCH_BATCH_SIZE
)

gaht_throughput_samples_sec = (
    BENCH_BATCH_SIZE /
    (gaht_latency_batch_ms / 1000)
)


# ------------------------------------------------------------
# 6. FINAL RESULTS
# ------------------------------------------------------------

gaht_benchmark_results = {
    "model": "GAHT",

    "accuracy": float(test_acc),
    "macro_f1": float(test_f1),

    "parameters": int(gaht_parameters),
    "parameters_m": float(gaht_parameters_m),

    "model_size_mb": float(
        gaht_model_size_mb
    ),

    "flops": float(gaht_flops),
    "gflops": float(gaht_gflops),

    "peak_gpu_memory_mb": float(
        gaht_peak_gpu_memory_mb
    ),

    "batch_size": BENCH_BATCH_SIZE,

    "latency_batch_ms": float(
        gaht_latency_batch_ms
    ),

    "latency_batch_median_ms": float(
        gaht_latency_batch_median_ms
    ),

    "latency_batch_p95_ms": float(
        gaht_latency_batch_p95_ms
    ),

    "latency_batch_std_ms": float(
        gaht_latency_batch_std_ms
    ),

    "latency_per_sample_ms": float(
        gaht_latency_per_sample_ms
    ),

    "throughput_samples_sec": float(
        gaht_throughput_samples_sec
    ),
}


print()
print("=" * 70)
print("GAHT BENCHMARK RESULTS")
print("=" * 70)

for key, value in gaht_benchmark_results.items():
    print(f"{key:30s}: {value}")

Trainable parameters: 972624
Parameters (M): 0.972624
Raw state-dict size: 3.714 MB
FLOPs   : 218593024.0
GFLOPs  : 0.218593024
Peak GPU memory: 57.93896484375 MB

GAHT BENCHMARK RESULTS
model                         : GAHT
accuracy                      : 0.9581180811808118
macro_f1                      : 0.942374614341102
parameters                    : 972624
parameters_m                  : 0.972624
model_size_mb                 : 3.7137069702148438
flops                         : 218593024.0
gflops                        : 0.218593024
peak_gpu_memory_mb            : 57.93896484375
batch_size                    : 64
latency_batch_ms              : 59.628575998794986
latency_batch_median_ms       : 55.874300000141375
latency_batch_p95_ms          : 85.56185500419815
latency_batch_std_ms          : 11.197986456608641
latency_per_sample_ms         : 0.9316964999811717
throughput_samples_sec        : 1073.3108904243052


In [37]:
# ============================================================
# SAVE MODEL RESULTS TO MASTER COMPARISON FILE
# ============================================================

from pathlib import Path
from openpyxl import load_workbook

MASTER_FILE = Path(
     r"results\salinas_sota_benchmark_results.xlsx"
)

# ------------------------------------------------------------
# PUT THE CURRENT MODEL'S BENCHMARK DICTIONARY HERE
# ------------------------------------------------------------

new_result = gaht_benchmark_results


# ------------------------------------------------------------
# LOAD MASTER FILE
# ------------------------------------------------------------

wb = load_workbook(MASTER_FILE)

ws = wb["Benchmark Results"]

headers = [
    cell.value
    for cell in ws[1]
]


# ------------------------------------------------------------
# READ EXISTING RESULTS
# ------------------------------------------------------------

rows = []

for row in ws.iter_rows(
    min_row=2,
    values_only=True,
):

    if not any(v is not None for v in row):
        continue

    result = dict(
        zip(headers, row)
    )

    rows.append(result)


# ------------------------------------------------------------
# UPDATE EXISTING MODEL OR APPEND NEW MODEL
# ------------------------------------------------------------

model_name = new_result["model"]

found = False

for i, result in enumerate(rows):

    if result["model"] == model_name:

        rows[i].update(new_result)

        found = True
        break


if not found:
    rows.append(new_result)


# ------------------------------------------------------------
# HYBRID MODEL = REFERENCE
# ------------------------------------------------------------

hybrid = next(
    r for r in rows
    if r["model"] == "Hybrid Spatial-Spectral"
)


# ------------------------------------------------------------
# COMPARISON METRICS
# ------------------------------------------------------------

for r in rows:

    r["params_vs_hybrid_pct"] = (
        (r["parameters"] /
         hybrid["parameters"] - 1)
        * 100
    )

    r["memory_vs_hybrid_pct"] = (
        (r["peak_gpu_memory_mb"] /
         hybrid["peak_gpu_memory_mb"] - 1)
        * 100
    )

    r["latency_vs_hybrid_pct"] = (
        (r["latency_per_sample_ms"] /
         hybrid["latency_per_sample_ms"] - 1)
        * 100
    )

    r["throughput_vs_hybrid_pct"] = (
        (r["throughput_samples_sec"] /
         hybrid["throughput_samples_sec"] - 1)
        * 100
    )

    r["accuracy_vs_hybrid_pp"] = (
        r["accuracy"] -
        hybrid["accuracy"]
    ) * 100

    r["macro_f1_vs_hybrid_pp"] = (
        r["macro_f1"] -
        hybrid["macro_f1"]
    ) * 100


# ------------------------------------------------------------
# RANKINGS
# ------------------------------------------------------------

accuracy_sorted = sorted(
    rows,
    key=lambda r: r["accuracy"],
    reverse=True
)

f1_sorted = sorted(
    rows,
    key=lambda r: r["macro_f1"],
    reverse=True
)

efficiency_sorted = sorted(
    rows,
    key=lambda r: r["latency_per_sample_ms"]
)


for rank, r in enumerate(
    accuracy_sorted, 1
):
    r["accuracy_rank"] = rank


for rank, r in enumerate(
    f1_sorted, 1
):
    r["macro_f1_rank"] = rank


for rank, r in enumerate(
    efficiency_sorted, 1
):
    r["efficiency_rank"] = rank


# ------------------------------------------------------------
# MAKE SURE ALL COLUMNS EXIST
# ------------------------------------------------------------

if "efficiency_rank" not in headers:
    headers.append("efficiency_rank")


# ------------------------------------------------------------
# REWRITE A SHEET
# ------------------------------------------------------------

def rewrite_sheet(
    sheet_name,
    ordered_rows,
):

    ws = wb[sheet_name]

    ws.delete_rows(
        1,
        ws.max_row
    )

    ws.append(headers)

    for r in ordered_rows:

        ws.append([
            r.get(h)
            for h in headers
        ])


# ------------------------------------------------------------
# UPDATE ALL COMPARISON SHEETS
# ------------------------------------------------------------

rewrite_sheet(
    "Benchmark Results",
    accuracy_sorted,
)

rewrite_sheet(
    "Accuracy Ranking",
    accuracy_sorted,
)

rewrite_sheet(
    "Macro-F1 Ranking",
    f1_sorted,
)

rewrite_sheet(
    "Efficiency Ranking",
    efficiency_sorted,
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

wb.save(MASTER_FILE)


# ------------------------------------------------------------
# CONFIRM
# ------------------------------------------------------------

print("=" * 70)
print("RESULT SAVED")
print("=" * 70)

print("Model:", model_name)
print("Accuracy:", new_result["accuracy"])
print("Macro-F1:", new_result["macro_f1"])
print("Parameters:", new_result["parameters"])
print("GFLOPs:", new_result["gflops"])

print()
print("Master file:")
print(MASTER_FILE)

print()
print("Models currently stored:")

for r in accuracy_sorted:
    print(
        f"  {r['model']:<30}"
        f" Acc={r['accuracy']:.4f}"
        f"  F1={r['macro_f1']:.4f}"
    )

RESULT SAVED
Model: GAHT
Accuracy: 0.9581180811808118
Macro-F1: 0.942374614341102
Parameters: 972624
GFLOPs: 0.218593024

Master file:
results\salinas_sota_benchmark_results.xlsx

Models currently stored:
  SSFTT (Official)               Acc=0.9815  F1=0.9541
  Hybrid Spatial-Spectral        Acc=0.9810  F1=0.9639
  GSC-ViT                        Acc=0.9806  F1=0.9667
  MorphFormer                    Acc=0.9708  F1=0.9619
  GAHT                           Acc=0.9581  F1=0.9424
  SpectralFormer (Official)      Acc=0.9274  F1=0.9187


In [36]:
from pathlib import Path

matches = list(
    Path(".").rglob("*.xlsx")
)

for p in matches:
    print(p)

results\salinas_sota_benchmark_results.xlsx


In [38]:
# ============================================================
# SAVE / UPDATE GAHT IN MASTER CSV
# ============================================================

from pathlib import Path
import pandas as pd

CSV_FILE = Path(
    r"results\salinas_sota_benchmark_results.csv"
)

new_result = gaht_benchmark_results

print("CSV:", CSV_FILE.resolve())
print("Exists:", CSV_FILE.exists())

# Load existing CSV
df = pd.read_csv(CSV_FILE)

# Remove old GAHT row if it already exists
# This prevents duplicates if we run the cell again.
if "model" in df.columns:
    df = df[
        df["model"].astype(str) != new_result["model"]
    ].copy()

# Add GAHT
new_row = pd.DataFrame([new_result])

df = pd.concat(
    [df, new_row],
    ignore_index=True,
)

# Optional: sort by accuracy
df = df.sort_values(
    "accuracy",
    ascending=False,
).reset_index(drop=True)

# Recalculate rankings
df["accuracy_rank"] = (
    df["accuracy"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df["macro_f1_rank"] = (
    df["macro_f1"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df["efficiency_rank"] = (
    df["latency_per_sample_ms"]
    .rank(method="min", ascending=True)
    .astype(int)
)

# Save back to SAME CSV
df.to_csv(
    CSV_FILE,
    index=False,
)

print()
print("=" * 70)
print("GAHT SAVED TO MASTER CSV")
print("=" * 70)
print("File:", CSV_FILE)
print("Models:", len(df))

print()
print(
    df[
        [
            "model",
            "accuracy",
            "macro_f1",
            "parameters",
            "gflops",
            "latency_per_sample_ms",
            "throughput_samples_sec",
        ]
    ].to_string(index=False)
)

CSV: C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\notebooks\results\salinas_sota_benchmark_results.csv
Exists: True

GAHT SAVED TO MASTER CSV
File: results\salinas_sota_benchmark_results.csv
Models: 6

                    model  accuracy  macro_f1  parameters   gflops  latency_per_sample_ms  throughput_samples_sec
         SSFTT (Official)  0.981500  0.954100      153224 0.016900               0.067400            14844.050000
  Hybrid Spatial-Spectral  0.981000  0.963900      605712 0.082100               0.342300             2921.360000
                  GSC-ViT  0.980627  0.966688      179024 1.694452               0.233398             4284.530302
              MorphFormer  0.970756  0.961936      280752 8.522273               1.074798              930.407821
                     GAHT  0.958118  0.942375      972624 0.218593               0.931696             1073.310890
SpectralFormer (Official)  0.927400  0.918700      39938